# 链式法则与自动微分（Chain Rule & Automatic Differentiation）

对应课程：`phases/01-math-foundations/05-chain-rule-and-autodiff`

> 链式法则是每一个能学习的神经网络背后的引擎。

本 notebook 把 `autodiff.py` 里的核心函数拆开：每个函数一组中文注释，后面跟一小段可运行实验。完整打印型 demo 仍在 `autodiff.py`。

**贯穿全课的模式：** 前向算值并记下局部导数；反向从 $\partial L/\partial y=1$ 出发，按拓扑逆序把梯度乘回去。


## 0. 依赖

只用标准库，和课程允许清单一致。


In [1]:
import math


## 1. 链式法则：复合函数的导数

若 $y = f(g(x))$，则

$$
\frac{dy}{dx} = \frac{dy}{dg}\cdot\frac{dg}{dx} = f'(g(x))\,g'(x)
$$

神经网络是成百上千个函数叠在一起。训练需要 $\partial L/\partial w$，手工推不可行，有限差分又太慢。反向模式自动微分：每个算子只提供自己的局部导数，链式法则由引擎代劳。


## 2. `Value`：计算图上的一个节点

每个节点存：

- `data`：前向算出的数
- `grad`：$\partial L / \partial \text{this}$，初始 0，反向时累加（`+=` 是为了处理同一个节点被用两次）
- `_prev`：子节点，用来建图
- `_backward`：闭包，用已算好的 `out.grad` 往子节点灌梯度


In [2]:
class Value:
    def __init__(self, data, children=(), op=''):
        # data: 前向值；grad: 反向累加；_prev: 构图；_op: 调试用算子名
        self.data = float(data)
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(children)
        self._op = op

    def __repr__(self):
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

    def __add__(self, other):
        # d(a+b)/da = 1，d(a+b)/db = 1；梯度原样传回两边
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')

        def _backward():
            self.grad += out.grad
            other.grad += out.grad

        out._backward = _backward
        return out

    def __radd__(self, other):
        return self.__add__(other)

    def __mul__(self, other):
        # d(ab)/da = b，d(ab)/db = a
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad

        out._backward = _backward
        return out

    def __rmul__(self, other):
        return self.__mul__(other)

    def __neg__(self):
        return self * -1

    def __sub__(self, other):
        return self + (-other)

    def __rsub__(self, other):
        return other + (-self)

    def __pow__(self, n):
        # d(x^n)/dx = n * x^(n-1)；n 当常数，不建图
        out = Value(self.data ** n, (self,), f'**{n}')

        def _backward():
            self.grad += n * (self.data ** (n - 1)) * out.grad

        out._backward = _backward
        return out

    def __truediv__(self, other):
        # a/b = a * b^(-1)
        return self * (other ** -1) if isinstance(other, Value) else self * (Value(other) ** -1)

    def relu(self):
        # relu'(x) = 1 if x>0 else 0；用 out.data 判断，避开对 0 的歧义
        out = Value(max(0, self.data), (self,), 'relu')

        def _backward():
            self.grad += (1.0 if out.data > 0 else 0.0) * out.grad

        out._backward = _backward
        return out

    def tanh(self):
        # d tanh / dx = 1 - tanh(x)^2
        t = math.tanh(self.data)
        out = Value(t, (self,), 'tanh')

        def _backward():
            self.grad += (1 - t ** 2) * out.grad

        out._backward = _backward
        return out

    def exp(self):
        # d(e^x)/dx = e^x，正好是前向值
        e = math.exp(self.data)
        out = Value(e, (self,), 'exp')

        def _backward():
            self.grad += e * out.grad

        out._backward = _backward
        return out

    def log(self):
        # d(ln x)/dx = 1/x
        out = Value(math.log(self.data), (self,), 'log')

        def _backward():
            self.grad += (1.0 / self.data) * out.grad

        out._backward = _backward
        return out

    def backward(self):
        # 反向模式：先拓扑排序保证子节点在父节点前面，再从输出灌 grad=1
        topo = []
        visited = set()

        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)

        build_topo(self)

        self.grad = 1.0
        for v in reversed(topo):
            v._backward()


print(Value(2.0) + Value(3.0))
print(Value(2.0) * Value(3.0))


Value(data=5.0000, grad=0.0000)
Value(data=6.0000, grad=0.0000)


## 3. 实验：`y = \mathrm{relu}(x_1 x_2 + 1)`

$$
y = \mathrm{relu}(x_1 x_2 + 1),\quad x_1=2,\; x_2=3
$$

前向：$x_1 x_2=6$，加 1 得 7，ReLU 原样输出 7。内部全为正，ReLU 局部导数是 1，所以

$$
\frac{\partial y}{\partial x_1}=x_2=3,\qquad \frac{\partial y}{\partial x_2}=x_1=2
$$


In [3]:
x1 = Value(2.0)
x2 = Value(3.0)
y = (x1 * x2 + 1).relu()
y.backward()

print("y =", y.data)
print("dy/dx1 =", x1.grad, "  (期望 3 = x2)")
print("dy/dx2 =", x2.grad, "  (期望 2 = x1)")


y = 7.0
dy/dx1 = 3.0   (期望 3 = x2)
dy/dx2 = 2.0   (期望 2 = x1)


## 4. 实验：幂次 $(x^2)' = 2x$

在 $x=3$ 处，$x^2=9$，导数 $2\cdot 3=6$。`**` 的局部导数是 $n x^{n-1}$。


In [4]:
x = Value(3.0)
y = x ** 2
y.backward()
print("x**2 =", y.data, "  (期望 9)")
print("d/dx x**2 =", x.grad, "  (期望 6)")


x**2 = 9.0   (期望 9)
d/dx x**2 = 6.0   (期望 6)


## 5. `tanh` / `exp` / `log` 局部导数

$$
\tanh'(x)=1-\tanh^2(x),\quad (e^x)'=e^x,\quad (\ln x)'=\frac{1}{x}
$$

各自建一个极小图，确认反向值和解析导数对得上。


In [5]:
x = Value(0.5)
y = x.tanh()
y.backward()
print("tanh(0.5) =", round(y.data, 6), "  d/dx =", round(x.grad, 6),
      "  解析", round(1 - math.tanh(0.5) ** 2, 6))

x = Value(2.0)
y = x.exp()
y.backward()
print("exp(2) =", round(y.data, 6), "  d/dx =", round(x.grad, 6),
      "  解析", round(math.exp(2.0), 6))

x = Value(3.0)
y = x.log()
y.backward()
print("log(3) =", round(y.data, 6), "  d/dx =", round(x.grad, 6),
      "  解析", round(1.0 / 3.0, 6))


tanh(0.5) = 0.462117   d/dx = 0.786448   解析 0.786448
exp(2) = 7.389056   d/dx = 7.389056   解析 7.389056
log(3) = 1.098612   d/dx = 0.333333   解析 0.333333


## 6. `gradient_check`：用中心差分验收 autograd

$$
f'(x)\approx \frac{f(x+h)-f(x-h)}{2h}
$$

自动微分给精确值（浮点误差内），有限差分给独立对照。两者差应在 $10^{-5}$ 量级以下。


In [6]:
def gradient_check(build_expr, x_val, h=1e-7):
    """同一表达式：autograd 一次，中心差分一次，返回 (ad, num, |差|)。"""
    x = Value(x_val)
    y = build_expr(x)
    y.backward()
    autodiff_grad = x.grad

    y_plus = build_expr(Value(x_val + h)).data
    y_minus = build_expr(Value(x_val - h)).data
    numerical_grad = (y_plus - y_minus) / (2 * h)

    diff = abs(autodiff_grad - numerical_grad)
    return autodiff_grad, numerical_grad, diff


ad, num, diff = gradient_check(lambda t: t ** 2, 3.0)
print("f(x) = x**2  at x=3")
print("autodiff :", ad)
print("numerical:", num)
print("|diff|   :", diff)


f(x) = x**2  at x=3
autodiff : 6.0
numerical: 5.999999990180527
|diff|   : 9.819473234529141e-09


## 对照表

| 函数 / 方法 | 角色 |
|------|------|
| `Value.__add__` | 局部导数 1，梯度原样分给两边 |
| `Value.__mul__` | 交叉相乘：一边的梯度乘另一边的值 |
| `Value.__pow__` | $n x^{n-1}$；指数当常数 |
| `Value.relu` | 正区透传，负区切断 |
| `Value.tanh` / `exp` / `log` | 常用激活与对数似然积木 |
| `Value.backward` | 拓扑排序 + 反向模式，输出处 seed=1 |
| `gradient_check` | 中心差分验收 autograd |

要看完整打印 demo（含 mini MLP / XOR），运行：

```bash
python autodiff.py
```
